In [ ]:
import pandas as pd

# 1. Cargar el CSV
df = pd.read_csv('/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/new_input_data_all.csv')
# 2. Crear la columna 'mask' (ruta base + lógica de carpetas)
base_path = "/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib/derivatives/aut_seg"
df['mask'] = df.apply(lambda row: f"{base_path}/sub-{row['subj']}/ses-{row['sess']}/mod-mr/anat/sub-{row['subj']}_ses-{row['sess']}_Sag_T1_disc_discs.nii.gz", axis=1)

# 3. PIVOTAR LA TABLA
# Usamos las columnas fijas como índice y 'disc' se convierte en columnas
cols_fijas = ['subj', 'sess', 'image_T2', 'image_T1', 'mask', 'XNAT_name']
df_pivot = df.pivot_table(index=cols_fijas, 
                          columns='disc', 
                          values='Pfirrmann', 
                          aggfunc='first').reset_index()

# 4. Renombrar las columnas al formato deseado
df_pivot = df_pivot.rename(columns={
    'subj': 'patient_id',
    'sess': 'study_id',
    'image_T2': 'T2',
    'image_T1': 'T1'
})

# 5. Asegurar que existan las 5 columnas de discos (aunque algún paciente no los tenga todos)
discos_deseados = ['L5-S1', 'L4-L5', 'L3-L4', 'L2-L3', 'L1-L2']

for disco in discos_deseados:
    if disco not in df_pivot.columns:
        df_pivot[disco] = None  # Crea la columna vacía si no existe en el CSV original

# 6. Reordenar las columnas finales
columnas_finales = ['patient_id', 'study_id', 'T2', 'T1', 'mask', 'XNAT_name'] + discos_deseados
df_final = df_pivot[columnas_finales]

# 7. Guardar
df_final.to_csv('/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/updated_patients.csv', index=False)

print("Proceso completado. Archivo generado: 'updated_patients.csv'")
print(df_final.head())

Proceso completado. Archivo generado: 'updated_patients.csv'
disc patient_id study_id                                                 T2  \
0           S02      E02  /mnt/datalake/openmind/Midas/training_wheels_S...   
1           S03      E03  /mnt/datalake/openmind/Midas/training_wheels_S...   
2           S04      E04  /mnt/datalake/openmind/Midas/training_wheels_S...   
3           S05      E05  /mnt/datalake/openmind/Midas/training_wheels_S...   
4           S06      E06  /mnt/datalake/openmind/Midas/training_wheels_S...   

disc                                                 T1  \
0     /mnt/datalake/openmind/Midas/training_wheels_S...   
1     /mnt/datalake/openmind/Midas/training_wheels_S...   
2     /mnt/datalake/openmind/Midas/training_wheels_S...   
3     /mnt/datalake/openmind/Midas/training_wheels_S...   
4     /mnt/datalake/openmind/Midas/training_wheels_S...   

disc                                               mask     XNAT_name L5-S1  \
0     /mnt/datalake/openmind/M

In [ ]:
# RUTAS DE ARCHIVOS
path_csv_1 = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/updated_patients.csv"
path_csv_2 = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/filtered_midas900_t2w_partition.csv"

# 1. Cargar los dos DataFrames
df1 = pd.read_csv(path_csv_1)
df2 = pd.read_csv(path_csv_2)

# 2. Preparar el df2 para que coincida con el df1
# Renombramos las columnas según tu mapeo
mapeo_columnas = {
    'ID': 'patient_id',
    'Session_MIDS': 'study_id',
    'Image': 'T2',
    'Mask': 'mask',
    'Subject_XNAT': 'XNAT_name', # Asumimos que esta columna corresponde a XNAT_name
    # Mapeo de discos (Pfirrmann)
    '1': 'L5-S1',
    '2': 'L4-L5',
    '3': 'L3-L4',
    '4': 'L2-L3',
    '5': 'L1-L2'
}

df2 = df2.rename(columns=mapeo_columnas)

# 3. Gestionar columnas faltantes en df2
# El df1 tiene 'T1', pero el df2 no parece tenerlo en tu descripción.
# Creamos la columna T1 vacía (o con NaN) para que la unión no falle.
if 'T1' not in df2.columns:
    df2['T1'] = None  # O pd.NA

# 4. Seleccionar y ordenar las columnas de df2 para que sean idénticas a df1
cols_finales = ['patient_id', 'study_id', 'T2', 'T1', 'mask', 'XNAT_name', 
                'L5-S1', 'L4-L5', 'L3-L4', 'L2-L3', 'L1-L2']

# Nos aseguramos de que df2 solo tenga estas columnas y en ese orden
df2_limpio = df2[cols_finales]

# Aseguramos también el orden en df1 (por precaución)
df1 = df1[cols_finales]

# 5. Unir (Concatenar) los dos DataFrames
df_total = pd.concat([df1, df2_limpio], ignore_index=True)

# 6. Guardar el resultado final
ruta_salida = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/updated_patients2.csv"
df_total.to_csv(ruta_salida, index=False)

print(f"Unión completada.")
print(f"Filas CSV 1: {len(df1)}")
print(f"Filas CSV 2: {len(df2)}")
print(f"Filas Totales: {len(df_total)}")
print(f"Archivo guardado en: {ruta_salida}")

Unión completada.
Filas CSV 1: 327
Filas CSV 2: 717
Filas Totales: 1044
Archivo guardado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/updated_patients2.csv
